# CosMx Multi-omics: Spatial Protein + RNA Integration — UC and CD (Fig 5)

1. RNA + protein normalization (geometric mean for protein)
2. OGN+RSPO3+ Fib niche immune composition (CD vs UC bar plot)
3. Fibronectin + FOXP3/CD68 dual-channel spatial maps

In [ ]:
# ── Paths ──
UC_DIR     = "/path/to/imputation_model/data/Spatial_Protein_CosMx"
CD_DIR     = "/path/to/cosmx_data/cd_multiomics"
OUTPUT_DIR = UC_DIR + "/figures"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scipy.stats as stats
from scipy.sparse import csr_matrix
import anndata as ad

In [ ]:
load_PATH = "/path/to/cosmx_data/cd_multiomics/Cos4_CD_M_batch1"

#Cos4_303i_312i
#Cos4_CD_M_batch1

In [ ]:
#rna='Cos4_303i_312i_RNA_2_7.26'

rna='Cos4_CD_M_batch1_RNA_1_7.26'

In [ ]:
#sample='303i_312i'
sample='CD_M_batch1'

In [ ]:
# === RNA ===

rna_path = os.path.join(load_PATH, rna, "Updated/FlatFiles")

print("Loading RNA expression matrix...")
rna_expr = pd.read_csv(rna_path+'/'+sample+"_exprMat_file.csv.gz")
rna_meta = pd.read_csv(rna_path+'/'+sample+"_metadata_file.csv.gz")
rna_expr=rna_expr.drop('fov',axis=1)

cell_id_col = [c for c in rna_expr.columns if 'cell' in c.lower()][0]
rna_cells = rna_expr[cell_id_col].values
gene_cols = [c for c in rna_expr.columns if c != cell_id_col]

# Drop negprobes, falsecodes, system controls
drop_prefixes = ('NegPrb', 'NegControl', 'SystemControl', 'FalseCode', 'Falsecode')
real_gene_cols = [c for c in gene_cols if not c.startswith(drop_prefixes)]

print(f"RNA: {len(rna_cells)} cells, {len(real_gene_cols)} genes (dropped {len(gene_cols) - len(real_gene_cols)} controls)")


In [ ]:

X_rna = csr_matrix(rna_expr[real_gene_cols].values)
adata_rna = ad.AnnData(X=X_rna)
adata_rna.obs_names = [str(c) for c in rna_cells]
adata_rna.var_names = real_gene_cols


In [ ]:
for col in rna_meta.columns:
    adata_rna.obs[col] = rna_meta[col].values

In [ ]:
pr='Cos4_CD_M_batch1_PR_1_7.22'
#Cos4_303i_312i_PR_2_7.22

In [ ]:
pr_pfix='Cos4CD_M1CD_M2PR1722'
#Cos4303i312iPR2722

In [ ]:
pr_path = os.path.join(load_PATH,pr, "FlatFiles")

print("\nLoading Protein expression matrix...")
pr_expr = pd.read_csv(pr_path+'/'+pr_pfix+"_exprMat_file.csv.gz")
pr_meta = pd.read_csv(pr_path+'/'+pr_pfix+"_metadata_file.csv.gz")
pr_expr=pr_expr.drop('fov',axis=1)
cell_id_col_pr = [c for c in pr_expr.columns if 'cell' in c.lower()][0]
pr_cells = pr_expr[cell_id_col_pr].values
protein_cols = [c for c in pr_expr.columns if c != cell_id_col_pr]


In [ ]:
neg_cols_pr = [c for c in protein_cols if 'neg' in c.lower() or 'control' in c.lower()]
real_protein_cols = [c for c in protein_cols if c not in neg_cols_pr]

print(f"Protein: {len(pr_cells)} cells, {len(real_protein_cols)} proteins (dropped {len(neg_cols_pr)} controls)")

X_pr = csr_matrix(pr_expr[real_protein_cols].values.astype(np.float32))
adata_pr = ad.AnnData(X=X_pr)
adata_pr.obs_names = [str(c) for c in pr_cells]
adata_pr.var_names = real_protein_cols


In [ ]:

for col in pr_meta.columns:
    adata_pr.obs[col] = pr_meta[col].values

In [ ]:
adata_pr_filter = adata_pr[adata_pr.obs['cell'].isin(adata_rna.obs['cell'])]

In [ ]:
adata_rna_qc = adata_rna.copy()

In [ ]:
sc.pp.calculate_qc_metrics(
    adata_rna_qc, inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata_rna_qc,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata_rna_qc, min_counts=200)
sc.pp.filter_cells(adata_rna_qc, min_genes=100)
sc.pp.filter_cells(adata_rna_qc, max_counts=3500)
sc.pp.filter_cells(adata_rna_qc, max_genes=2000)
adata_rna_qc=adata_rna_qc[adata_rna_qc.obs['Area']<=30000]

In [ ]:
sc.pl.violin(
    adata_rna_qc,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_rna_qc.layers['raw']=adata_rna_qc.X

In [ ]:
sc.pp.normalize_total(adata_rna_qc, inplace=True)

In [ ]:
rna1_df=pd.DataFrame(adata_rna_qc.X.toarray())


In [ ]:
rna1_df.index = adata_rna_qc.obs['cell_id'].tolist()
rna1_df.columns = adata_rna_qc.var_names.tolist()


In [ ]:
rna1_df.to_csv(load_PATH+'/UPD_norm_nolog_rna.csv')

In [ ]:
rna1_anno = pd.read_csv(load_PATH+'/aucell_anno_upd.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
rna1_anno['label_clean'] = rna1_anno['label'].str.rstrip()

In [ ]:
rna1_anno_map=rna1_anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
adata_rna_qc.obs['cell_type'] = rna1_anno_map['cell type'].tolist()
adata_rna_qc.obs['cell_type_short'] = rna1_anno_map['cell type short'].tolist()
adata_rna_qc.obs['cell_category'] = rna1_anno_map['cell category'].tolist()
adata_rna_qc.obs['cell_type_general'] = rna1_anno_map['cell type general'].tolist()

In [ ]:
adata_rna_qc.obsm['spatial_fov']=adata_rna_qc.obs[['CenterX_global_px', 'CenterY_global_px']].values.astype(int)

In [ ]:
adata_rna_qc=adata_rna_qc[adata_rna_qc.obs['cell_category']!='Cycling']

In [ ]:
adata_rna_qc_312 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_312

In [ ]:
adata_rna_qc_375 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_375

In [ ]:
adata_pr_filter = adata_pr_filter[adata_pr_filter.obs['cell_id'].isin(adata_rna_qc.obs['cell_id'].tolist())]
adata_pr_filter

In [ ]:
adata_pr_filter_geo=adata_pr_filter.copy()

In [ ]:
# Step 1: Log-transform if necessary (for count data)
sc.pp.log1p(adata_pr_filter_geo)

# Step 2: Compute the geometric mean for each cell (margin=1, along rows)
geometric_mean = np.exp(np.mean(np.log(adata_pr_filter_geo.X.toarray() + 1), axis=1))

# Step 3: Normalize by dividing each value by the geometric mean (center each cell)
adata_pr_filter_geo.X = adata_pr_filter_geo.X / geometric_mean[:, np.newaxis]


In [ ]:
pro1_geo = pd.DataFrame(adata_pr_filter_geo.X.toarray())
pro1_geo.index = adata_pr_filter_geo.obs['cell_id'].tolist()
pro1_geo.columns = adata_pr_filter_geo.var_names.tolist()

In [ ]:
pro1_geo.columns = [i+'_geo' for i in pro1_geo.columns.tolist()]

In [ ]:
adata_rna_qc.obs = adata_rna_qc.obs.merge(pro1_geo, left_on = 'cell_id', right_index=True)

In [ ]:
adata_rna_qc_312 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_312

In [ ]:
adata_rna_qc_375 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_375

In [ ]:
rna2_pr = adata_rna_qc_375.obs[['cell_id','4-1BB_geo', 'B7-H3_geo', 'Bcl-2_geo', 'Beta-catenin_geo', 'CCR7_geo', 'CD11b_geo', 'CD11c_geo', 'CD123_geo', 'CD127_geo', 'CD138_geo', 'CD14_geo', 'CD15_geo', 'CD16_geo', 'CD163_geo', 'CD19_geo', 'CD20_geo', 'CD27_geo', 'CD3_geo', 'CD31_geo', 'CD34_geo', 'CD38_geo', 'CD39_geo', 'CD4_geo', 'CD40_geo', 'CD45_geo', 'CD45RA_geo', 'CD56_geo', 'CD68_geo', 'CD8_geo', 'CTLA4_geo', 'Channel-CD3_geo', 'Channel-CD45_geo', 'Channel-DNA_geo', 'Channel-Membrane_geo', 'Channel-PanCK_geo', 'EGFR_geo', 'EpCAM_geo', 'FABP4_geo', 'FOXP3_geo', 'Fibronectin_geo', 'GITR_geo', 'GZMA_geo', 'GZMB_geo', 'HLA-DR_geo', 'Her2_geo', 'ICAM1_geo', 'ICOS_geo', 'IDO1_geo', 'IL-18_geo', 'IL-1b_geo', 'IgD_geo', 'Ki-67_geo', 'LAG3_geo', 'LAMP1_geo', 'NF-kB p65_geo', 'PD-1_geo', 'PD-L1_geo', 'PD-L2_geo', 'SMA_geo', 'STING_geo', 'TCF7_geo', 'Tim-3_geo', 'VISTA_geo', 'Vimentin_geo', 'iNOS_geo', 'p53_geo', 'pan-RAS_geo', 'Ms IgG1_geo', 'Rb IgG_geo','cell_category']]
rna2_pr.index = rna2_pr['cell_id'].tolist()
rna2_pr=rna2_pr.drop('cell_id',axis=1)
rna2_pr

In [ ]:
adata_rna_qc.write_h5ad(f"{load_PATH}/rna_norm_pr_geonorm_not_upd_anno.h5ad")

In [ ]:
adata_rna_qc.write_h5ad(f"{load_PATH}/rna_norm_pr_geonorm_upd_anno.h5ad")

In [ ]:
adata_rna_qc=sc.read_h5ad(f"{load_PATH}/rna_norm_pr_geonorm_upd_anno.h5ad")

In [ ]:
adata_rna_qc_375 = adata_rna_qc[(adata_rna_qc.obs['fov'] <=29)].copy()
adata_rna_qc_375

In [ ]:
load_PATH = "/path/to/imputation_model/data/Spatial_Protein_CosMx"
adata_rna_qc_385=sc.read_h5ad(f"{load_PATH}/RNA1_Cos4_UC_M_batch1_S2_norm_wpr_anno_jd.h5ad")

In [ ]:
print(adata_rna_qc_375_sub.obs[['Area', 'Area.um2']].head(10))

In [ ]:
px_per_um = np.sqrt(3266 / 47.25)  # pixels per micron
distance_threshold = 50 * px_per_um  # 50 microns in pixels

In [ ]:
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

px_per_um = np.sqrt(3266 / 47.25)
radius_px = 100 * px_per_um

treg_niche_proteins = {
    'FOXP3_geo': 'FOXP3 (General/Early Treg)',
    'CTLA4_geo': 'CTLA4 (Early Treg)',
    'CD8_geo': 'CD8 (T Effector/Late Treg)',
    'Beta-catenin_geo': 'B-Catenin (Treg Transition)',
}
mac_niche_proteins = {
    'CD68_geo': 'CD68 (Mac)',
    'iNOS_geo': 'iNOS (Inflam Mac)',
    'IL-1b_geo': 'IL-1B (Inflam Mac)',
    'CD163_geo': 'CD163 (Res Mac)',
}

results = {}

for condition, (adata, ct_col, cat_col) in [
    ('CD', (adata_rna_qc_375_sub, 'cell_type_short', 'cell_category')),
    ('UC', (adata_rna_qc_385, 'aucell_cell_type_short', 'aucell_cell_category_from_type'))
]:
    spatial = adata.obsm['spatial_fov']
    fib_mask = (adata.obs[ct_col] == 'OGN+RSPO3+ Fib').values
    fib_coords = spatial[fib_mask]
    
    t_mask = (adata.obs[cat_col] == 'T').values
    myeloid_mask = (adata.obs[cat_col] == 'Myeloid').values
    
    tree_t = cKDTree(spatial[t_mask])
    tree_myeloid = cKDTree(spatial[myeloid_mask])
    
    t_global_indices = np.where(t_mask)[0]
    myeloid_global_indices = np.where(myeloid_mask)[0]
    
    t_neighbors = tree_t.query_ball_point(fib_coords, r=radius_px)
    for protein, label in treg_niche_proteins.items():
        if protein not in adata.obs.columns:
            continue
        all_vals = adata.obs[protein].values.astype(float)
        niche_means = []
        for neighbors in t_neighbors:
            if len(neighbors) > 0:
                global_idx = t_global_indices[neighbors]
                niche_means.append(np.mean(all_vals[global_idx]))
        results[(condition, label)] = np.array(niche_means)
    
    myeloid_neighbors = tree_myeloid.query_ball_point(fib_coords, r=radius_px)
    for protein, label in mac_niche_proteins.items():
        if protein not in adata.obs.columns:
            continue
        all_vals = adata.obs[protein].values.astype(float)
        niche_means = []
        for neighbors in myeloid_neighbors:
            if len(neighbors) > 0:
                global_idx = myeloid_global_indices[neighbors]
                niche_means.append(np.mean(all_vals[global_idx]))
        results[(condition, label)] = np.array(niche_means)

# Collect all p-values first
all_proteins = {**treg_niche_proteins, **mac_niche_proteins}
pvals = []
valid_indices = []

for i, (protein, label) in enumerate(all_proteins.items()):
    cd_vals = results.get(('CD', label), np.array([]))
    uc_vals = results.get(('UC', label), np.array([]))
    if len(cd_vals) >= 3 and len(uc_vals) >= 3:
        _, pval = mannwhitneyu(cd_vals, uc_vals, alternative='two-sided')
        pvals.append(pval)
        valid_indices.append(i)

_, qvals, _, _ = multipletests(pvals, method='fdr_bh')
qval_map = dict(zip(valid_indices, qvals))

fig, axes = plt.subplots(2, 4, figsize=(12,6))
fig.suptitle('')

# Row 1: Treg proteins
for i, (protein, label) in enumerate(treg_niche_proteins.items()):
    ax = axes[0, i]
    cd_vals = results.get(('CD', label), np.array([]))
    uc_vals = results.get(('UC', label), np.array([]))
    
    if len(cd_vals) < 3 or len(uc_vals) < 3:
        ax.set_title(f'{label}\n(insufficient)', fontsize=9)
        ax.axis('off')
        continue
    
    bp = ax.boxplot([cd_vals, uc_vals], labels=['CD', 'UC'], widths=0.5,
                   patch_artist=True, showfliers=False)
    bp['boxes'][0].set_facecolor('#E63946')
    bp['boxes'][1].set_facecolor('#457B9D')
    
    idx = list(all_proteins.values()).index(label)
    qval = qval_map.get(idx, 1.0)
    cohens_d = (np.mean(cd_vals) - np.mean(uc_vals)) / np.sqrt((np.std(cd_vals)**2 + np.std(uc_vals)**2) / 2)
    sig = '***' if qval < 0.001 else '**' if qval < 0.01 else '*' if qval < 0.05 else 'ns'
    
    ax.set_title(f'{label}\n{sig} q={qval:.4f}, d={cohens_d:.2f}', fontsize=12)
    if i == 0:
        ax.set_ylabel('Treg PR Markers\n Near OGN+RSPO3+ Fib', fontsize=11)

# Row 2: Mac proteins
for i, (protein, label) in enumerate(mac_niche_proteins.items()):
    ax = axes[1, i]
    cd_vals = results.get(('CD', label), np.array([]))
    uc_vals = results.get(('UC', label), np.array([]))
    
    if len(cd_vals) < 3 or len(uc_vals) < 3:
        ax.set_title(f'{label}\n(insufficient)', fontsize=9)
        ax.axis('off')
        continue
    
    bp = ax.boxplot([cd_vals, uc_vals], labels=['CD', 'UC'], widths=0.5,
                   patch_artist=True, showfliers=False)
    bp['boxes'][0].set_facecolor('#E63946')
    bp['boxes'][1].set_facecolor('#457B9D')
    
    idx = list(all_proteins.values()).index(label)
    qval = qval_map.get(idx, 1.0)
    cohens_d = (np.mean(cd_vals) - np.mean(uc_vals)) / np.sqrt((np.std(cd_vals)**2 + np.std(uc_vals)**2) / 2)
    sig = '***' if qval < 0.001 else '**' if qval < 0.01 else '*' if qval < 0.05 else 'ns'
    
    ax.set_title(f'{label}\n{sig} q={qval:.4f}, d={cohens_d:.2f}', fontsize=12)
    if i == 0:
        ax.set_ylabel('Mac PR Markers\n Near OGN+RSPO3+ Fib', fontsize=11)

plt.tight_layout()
plt.savefig('OGN_Fib_niche_protein_CD_vs_UC_FDR.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def protein_annotate(adata):
    obs = adata.obs
    labels = np.full(len(obs), 'Other', dtype=object)
    
    # Get protein values
    cd45 = obs['CD45_geo'].values.astype(float)
    cd3 = obs['CD3_geo'].values.astype(float)
    cd4 = obs['CD4_geo'].values.astype(float)
    cd8 = obs['CD8_geo'].values.astype(float)
    foxp3 = obs['FOXP3_geo'].values.astype(float)
    cd68 = obs['CD68_geo'].values.astype(float)
    cd163 = obs['CD163_geo'].values.astype(float)
    cd19 = obs['CD19_geo'].values.astype(float)
    cd20 = obs['CD20_geo'].values.astype(float)
    cd138 = obs['CD138_geo'].values.astype(float)
    cd56 = obs['CD56_geo'].values.astype(float)
    epcam = obs['EpCAM_geo'].values.astype(float)
    vim = obs['Vimentin_geo'].values.astype(float)
    sma = obs['SMA_geo'].values.astype(float)
    cd31 = obs['CD31_geo'].values.astype(float)
    
    # Use per-marker median as threshold
    def high(vals):
        return vals > np.median(vals)
    
    # Epithelial: EpCAM high, CD45 low
    labels[(high(epcam)) & (~high(cd45))] = 'Epithelial'
    
    # Endothelial: CD31 high, CD45 low, EpCAM low
    labels[(high(cd31)) & (~high(cd45)) & (~high(epcam))] = 'Endothelial'
    
    # Fibroblast: Vimentin high, CD45 low, EpCAM low, CD31 low
    labels[(high(vim)) & (~high(cd45)) & (~high(epcam)) & (~high(cd31))] = 'Fibroblast'
    
    # Myofibroblast: SMA high, CD45 low, EpCAM low
    labels[(high(sma)) & (~high(cd45)) & (~high(epcam)) & (labels == 'Fibroblast')] = 'Myofibroblast'
    
    # Immune cells: CD45 high
    immune = high(cd45)
    
    # T cells: CD45+CD3+
    t_cell = immune & high(cd3)
    labels[t_cell & high(cd4) & ~high(foxp3)] = 'CD4 T cell'
    labels[t_cell & high(cd4) & high(foxp3)] = 'Treg'
    labels[t_cell & high(cd8)] = 'CD8 T cell'
    labels[t_cell & ~high(cd4) & ~high(cd8)] = 'T cell (other)'
    
    # Macrophage: CD45+CD68+
    labels[immune & high(cd68) & ~high(cd3)] = 'Inflam Mac'
    labels[immune & high(cd68) & high(cd163) & ~high(cd3)] = 'Res Mac'
    
    # B cell: CD45+CD19/CD20+
    labels[immune & (high(cd19) | high(cd20)) & ~high(cd3) & ~high(cd68)] = 'B cell'
    
    # Plasma: CD138+
    labels[immune & high(cd138) & ~high(cd3) & ~high(cd68)] = 'Plasma'
    
    # NK: CD56+CD3-
    labels[immune & high(cd56) & ~high(cd3)] = 'NK cell'
    
    return labels

# Apply
for condition, (adata, ct_col) in [('CD', (adata_rna_qc_375_sub, 'cell_type_short')),
                                     ('UC', (adata_rna_qc_385, 'aucell_cell_type_short'))]:
    adata.obs['protein_celltype'] = protein_annotate(adata)
    print(f"\n{condition}:")
    print(adata.obs['protein_celltype'].value_counts())

# Spatial plot
colors = {
    'Epithelial': '#F4A460', 'Endothelial': '#FF69B4', 
    'Fibroblast': '#E63946', 'Myofibroblast': '#A31621',
    'CD4 T cell': '#457B9D', 'CD8 T cell': '#1D3557', 'Treg': '#2EC4B6',
    'T cell (other)': '#A8DADC',
    'Inflam Mac': '#1B4332', 'Res Mac': '#52B788',
    'B cell': '#E9C46A', 'Plasma': '#F4A261', 'NK cell': '#9B59B6',
    'Other': '#E8E8E8',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))


for i, (condition, (adata, ct_col)) in enumerate([
    ('CD', (adata_rna_qc_375_sub, 'cell_type_short')),
    ('UC', (adata_rna_qc_385, 'aucell_cell_type_short'))
]):
    ax = axes[i]
    x = adata.obsm['spatial_fov'][:, 0]
    y = adata.obsm['spatial_fov'][:, 1]
    
    other_mask = adata.obs['protein_celltype'] == 'Other'
    ax.scatter(x[other_mask], y[other_mask], c='#E8E8E8', s=1, alpha=0.2, rasterized=True)
    
    handles = []
    for ct, color in colors.items():
        if ct == 'Other':
            continue
        mask = (adata.obs['protein_celltype'] == ct).values
        if mask.sum() > 0:
            sc = ax.scatter(x[mask], y[mask], c=color, s=1.75, alpha=0.7, label=f'{ct} ({mask.sum()})', rasterized=True)
            handles.append(sc)

    ax.set_title(f'{condition}', fontsize=12)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.savefig('protein_based_annotation_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import numpy as np

px_per_um = np.sqrt(3266 / 47.25)
radius_px = 100 * px_per_um

fig, ax = plt.subplots(figsize=(6,4))

immune_types = ['CD4 T cell', 'CD8 T cell', 'Suppressive Treg', 'T cell (other)', 
                'Inflam Mac', 'Res Mac', 'B cell', 'Plasma', 'NK cell']

for condition, (adata, ct_col) in [('CD', (adata_rna_qc_375_sub, 'cell_type_short')),
                                     ('UC', (adata_rna_qc_385, 'aucell_cell_type_short'))]:
    spatial = adata.obsm['spatial_fov']
    fib_mask = (adata.obs[ct_col] == 'OGN+RSPO3+ Fib').values
    fib_coords = spatial[fib_mask]
    
    tree = cKDTree(spatial)
    neighbors = tree.query_ball_point(fib_coords, r=radius_px)
    
    # Pool all neighbor indices
    all_neighbors = set()
    for n in neighbors:
        all_neighbors.update(n)
    all_neighbors = list(all_neighbors)
    
    # Get protein cell types of neighbors
    neighbor_types = adata.obs['protein_celltype'].values[all_neighbors]
    
    props = {}
    total_immune = sum(1 for t in neighbor_types if t in immune_types)
    for it in immune_types:
        count = sum(1 for t in neighbor_types if t == it)
        props[it] = count / total_immune * 100 if total_immune > 0 else 0
    
    print(f"\n{condition} — Immune composition in OGN+RSPO3+ Fib niche:")
    for it, pct in props.items():
        print(f"  {it}: {pct:.1f}%")

# Grouped bar plot
x = np.arange(len(immune_types))
width = 0.35

cd_props = []
uc_props = []

for condition, (adata, ct_col) in [('CD', (adata_rna_qc_375_sub, 'cell_type_short')),
                                     ('UC', (adata_rna_qc_385, 'aucell_cell_type_short'))]:
    spatial = adata.obsm['spatial_fov']
    fib_mask = (adata.obs[ct_col] == 'OGN+RSPO3+ Fib').values
    fib_coords = spatial[fib_mask]
    
    tree = cKDTree(spatial)
    neighbors = tree.query_ball_point(fib_coords, r=radius_px)
    all_neighbors = list(set(n for ns in neighbors for n in ns))
    neighbor_types = adata.obs['protein_celltype'].values[all_neighbors]
    total_immune = sum(1 for t in neighbor_types if t in immune_types)
    
    props = []
    for it in immune_types:
        count = sum(1 for t in neighbor_types if t == it)
        props.append(count / total_immune * 100 if total_immune > 0 else 0)
    
    if condition == 'CD':
        cd_props = props
    else:
        uc_props = props

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(x - width/2, cd_props, width, label='CD', color='#E63946')
ax.bar(x + width/2, uc_props, width, label='UC', color='#457B9D')
ax.set_xticks(x)
ax.set_xticklabels(immune_types, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('%Cells in OGN+RSPO3+ Fib Niche', fontsize=10)
ax.set_xlabel('Immune Cells by PR Markers', fontsize=10)
ax.set_title('')
ax.legend()

plt.savefig('fib_niche_immune_composition.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

fig, axes = plt.subplots(2, 2, figsize=(16, 14), facecolor='black')
fig.patch.set_facecolor('black')

# Compute shared scales across both datasets
def get_shared_range(adatas, protein, lo=5, hi=95):
    all_vals = np.concatenate([a.obs[protein].values.astype(float) for a in adatas])
    return np.percentile(all_vals[~np.isnan(all_vals)], [lo, hi])

adatas = [adata_rna_qc_375_sub, adata_rna_qc_385]
fib_range = get_shared_range(adatas, 'Fibronectin_geo')
foxp3_range = get_shared_range(adatas, 'FOXP3_geo')
cd68_range = get_shared_range(adatas, 'CD68_geo')

shared_ranges = {
    'Fibronectin_geo': fib_range,
    'FOXP3_geo': foxp3_range,
    'CD68_geo': cd68_range,
}

# Minimum signal threshold — cells below this in BOTH channels are greyed out
signal_thresh = 0.15  # normalized, so bottom 15% of both = no plot

plot_configs = [
    (axes[0, 0], adata_rna_qc_375_sub, 'CD — Fibronectin + FOXP3', 'Fibronectin_geo', 'FOXP3_geo'),
    (axes[0, 1], adata_rna_qc_375_sub, 'CD — Fibronectin + CD68', 'Fibronectin_geo', 'CD68_geo'),
    (axes[1, 0], adata_rna_qc_385, 'UC — Fibronectin + FOXP3', 'Fibronectin_geo', 'FOXP3_geo'),
    (axes[1, 1], adata_rna_qc_385, 'UC — Fibronectin + CD68', 'Fibronectin_geo', 'CD68_geo'),
]

for ax, adata, title, prot1, prot2 in plot_configs:
    spatial = adata.obsm['spatial_fov']
    x, y = spatial[:, 0], spatial[:, 1]
    
    vals1 = adata.obs[prot1].values.astype(float)
    vals2 = adata.obs[prot2].values.astype(float)
    
    v1min, v1max = shared_ranges[prot1]
    v2min, v2max = shared_ranges[prot2]
    
    norm1 = np.clip((vals1 - v1min) / (v1max - v1min), 0, 1)
    norm2 = np.clip((vals2 - v2min) / (v2max - v2min), 0, 1)
    
    signal_mask = (norm1 > signal_thresh) | (norm2 > signal_thresh)
    
    # Black background
    ax.set_facecolor('black')
    
    # Dim tissue outline for low-signal cells
    ax.scatter(x[~signal_mask], y[~signal_mask], c='#1a1a1a', s=0.5, alpha=0.2, rasterized=True)
    
    # Signal cells
    # Signal cells
    rgba = np.zeros((signal_mask.sum(), 4))
    rgba[:, 1] = norm1[signal_mask] ** 0.8  # green = fibronectin
    
    if 'CD68' in prot2:
        rgba[:, 0] = norm2[signal_mask] ** 0.8 * 0.7
        rgba[:, 2] = norm2[signal_mask] ** 0.8
    else:
        rgba[:, 0] = norm2[signal_mask] ** 0.8
    
    # ADD THIS — alpha channel
    rgba[:, 3] = np.clip(np.maximum(norm1[signal_mask], norm2[signal_mask]) * 0.95 + 0.05, 0, 1)
    
    ax.scatter(x[signal_mask], y[signal_mask], c=rgba, s=3, rasterized=True)

        
    prot1_label = prot1.replace('_geo', '')
    prot2_label = prot2.replace('_geo', '')

    from sklearn.cluster import DBSCAN
    from matplotlib.patches import Circle

   # DBSCAN circles for co-localization zones
    # Find cells where BOTH channels are above threshold
    both_high = signal_mask & (norm1 > 0.4) & (norm2 > 0.4)
    high_coords = spatial[both_high]
    
    if len(high_coords) > 5:
        clustering = DBSCAN(eps=radius_px * 0.5, min_samples=5).fit(high_coords)
        max_circle_r = radius_px * 1.5
        
        for cl in set(clustering.labels_):
            if cl == -1:
                continue
            cl_points = high_coords[clustering.labels_ == cl]
            if len(cl_points) <8:
                continue
            cx, cy = cl_points[:, 0].mean(), cl_points[:, 1].mean()
            r = np.max(np.sqrt((cl_points[:, 0] - cx)**2 + (cl_points[:, 1] - cy)**2)) + radius_px * 0.5
            r = min(r, max_circle_r)
            ax.add_patch(Circle((cx, cy), r, fill=False, edgecolor='white',
                               linewidth=2.5, linestyle='-', alpha=0.8, zorder=4))
            
    sm1 = cm.ScalarMappable(cmap='Greens', norm=Normalize(vmin=v1min, vmax=v1max))
    
    if 'CD68' in prot2:
        sm2 = cm.ScalarMappable(cmap='Purples', norm=Normalize(vmin=v2min, vmax=v2max))
        prot2_color = '#BB88FF'
    else:
        sm2 = cm.ScalarMappable(cmap='Reds', norm=Normalize(vmin=v2min, vmax=v2max))
        prot2_color = '#FF4444'
    
    cb1 = plt.colorbar(sm1, ax=ax, shrink=0.4, location='left', pad=-0.02)
    cb1.set_label(prot1_label, fontsize=13, color='#00FF00')
    cb1.outline.set_edgecolor('white')
    cb1.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb1.ax.yaxis.get_ticklabels(), color='white')
    
    cb2 = plt.colorbar(sm2, ax=ax, shrink=0.4, location='right', pad=-0.02)
    cb2.set_label(prot2_label, fontsize=13, color=prot2_color)
    cb2.ax.yaxis.set_tick_params(color='white')
    cb2.outline.set_edgecolor('white')
    plt.setp(cb2.ax.yaxis.get_ticklabels(), color='white')
    
    ax.set_title(f'{title}',
                 fontsize=15, color='white')
    ax.set_xticks([])
    ax.set_yticks([])
    
plt.tight_layout()
plt.savefig('dual_protein_heatmap_CD_UC_shared.png', dpi=300, bbox_inches='tight')
plt.show()